# Cómo terminan los procedimientos de la AEPD

Cuaderno de lectura de la medición `aepd-correctivos/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos). Resoluciones de los procedimientos sancionadores y de apercibimiento de la Agencia Española de Protección de Datos, de expedientes de 2024 a 2026, clasificadas por su parte dispositiva. Carga los ficheros de al lado (o los descarga del repositorio si se ejecuta fuera de él), muestra la ficha de procedencia y dibuja una figura con matplotlib a secas. Solo lee; no repite la medición: para eso está `generar.py`.

In [ ]:
import urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/aepd-correctivos/"

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    if p.exists():
        return pd.read_csv(p, **kw)
    return pd.read_csv(RAW + nombre, **kw)

def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre, timeout=30) as r:
        return r.read().decode("utf-8")

## Ficha de procedencia

In [ ]:
print(texto("INSTANTANEA.md"))

## El dato

In [ ]:
res = leer("salidas/resoluciones.csv")
res = res[res.desenlace != "revocacion"]
ps = res[res.tipo.str.contains("PS")]
c24 = ps[ps.anio_expediente == 2024]
print("procedimientos sancionadores de 2024:", len(c24))
print((100 * c24.desenlace.value_counts(normalize=True)).round(1).to_string())
privados = c24[c24.regimen_art77 == 0]
eco = privados.desenlace.isin(["multa", "terminacion_reducida"]).mean()
print("sin administraciones:", len(privados), "· con sanción económica:", round(100 * eco, 1), "%")
imp = ps[ps.importe_final.notna()]
print("sanciones con importe:", len(imp), "· mediana:", imp.importe_final.median(), "€")
print(ps[ps.desenlace == "terminacion_reducida"].reduccion_pct.value_counts().to_string())

## Una figura

In [ ]:
serie = leer("salidas/serie_tipos.csv")
tabla = serie.pivot_table(index="anio", columns="tipo", values="resoluciones", aggfunc="sum", fill_value=0)
ax = tabla.plot.bar(stacked=True, figsize=(10, 4.5), width=0.8, legend=False)
ax.set_xlabel("año de firma")
ax.set_ylabel("resoluciones publicadas")
ax.set_title("Resoluciones de la AEPD por tipo de procedimiento")
ax.legend(fontsize=7, ncol=2, frameon=False)
plt.tight_layout()
plt.show()